In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
import os

properties = ["ehull", "formation_energy_peratom", "magmom_outcar","spillage", "mbj_bandgap", "Tc_supercon", "slme"]
for p in properties:
    # df = pd.read_csv(f"deterministics/tensorflow/alignn_matbert-base-cased_robo_prop_{p}_pred_otf.csv")
    path = f"pred/alignn_bert-base-uncased_robo_prop_{names[i]}_pred_otf.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)

    # df = pd.read_csv("pred/alignn_matbert-base-cased_robo_prop_Tc_supercon_pred_otf.csv")

    # df = pd.read_csv("pred/atomgpt_new/alignn_bert-base-uncased_robo_prop_spillage_pred_otf.csv")
    # df = pd.read_csv(f"pred/alignn_bert-base-uncased_robo_prop_spillage_pred_otf.csv")
    # Specify the column names
        true_col = 'labels'     # Replace with actual column name for ground truth
        pred_col = 'predictions'     # Replace with actual column name for predictions

        # Compute MAE
        mae = mean_absolute_error(df[true_col], df[pred_col])
    print(f"{names[i]} Mean Absolute Error (MAE): {mae:.6f}")

#### Framework-level Comparison

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# properties = ["formation_energy_peratom", "ehull", "magmom_outcar", "mbj_bandgap", "spillage", "slme", "Tc_supercon"]
properties = ["mbj_bandgap"]
# names = ["Formation Energy", "Energy above the Hull", "Magnetic Moment", "Bandgap_mBJ", "Spillage", "SLME", "Tc_Supercon"]
names = ["Bandgap_mBJ"]
# units = ['eV/atom', 'eV/atom', 'μB', 'eV', '', '%', 'K']
units = ['eV']

for i, p in enumerate(properties):
    # Load predictions
    df_tf = pd.read_csv(f"deterministics/tensorflow/alignn_matbert-base-cased_robo_prop_{p}_pred_otf.csv", index_col=0)
    df_pt = pd.read_csv(f"deterministics/prediction_torch/alignn_matbert-base-cased_robo_prop_mbj_bandgap_pred_otf_20250614_184558_gpu.csv")

    # Optional filtering
    # if p == "magmom_outcar":
    #     mask = df_tf["labels"] != 0
    #     df_tf = df_tf[mask]
    #     df_pt = df_pt[mask]

    # Add predictions
    df = pd.DataFrame({
        "TF": df_tf["predictions"].astype(float),
        "PT": df_pt["predictions"].astype(float)
    })
    mad_value = np.mean(np.abs(df["TF"] - df["PT"]))
    tf_mae_value = np.mean(np.abs(df_tf["predictions"] - df_tf["labels"]))
    pt_mae_value = np.mean(np.abs(df_pt["predictions"] - df_pt["labels"]))  
    print(f"{p} Mean Absolute Deviation (MAD): {mad_value:.4f}")
    print(f"{p} Tensorflow Mean Absolute Error (MAE): {tf_mae_value:.4f}")
    print(f"{p} PyTorch Mean Absolute Error (MAE): {pt_mae_value:.4f}")
    # Create JointGrid
    g = sns.JointGrid(data=df, x="PT", y="TF", height=6, ratio=3)

    # Central scatter plot
    g.plot_joint(sns.scatterplot, color="#e36c55", alpha=0.6, s=40, marker='o')

    # Add y = x reference line
    min_val = min(df["TF"].min(), df["PT"].min())
    max_val = max(df["TF"].max(), df["PT"].max())
    g.ax_joint.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

    g.ax_joint.set_xlim(min_val, max_val)
    g.ax_joint.set_ylim(min_val, max_val)



    # Force identical ticks
    ticks = g.ax_joint.get_xticks()
    if p in ["Tc_supercon", "spillage", "mbj_bandgap", "slme"]:
        # Force first tick to be zero (or start from zero if zero is in the list)
        ticks = np.array(ticks)
        zero_index = np.where(ticks == 0)[0]
        if len(zero_index) > 0:
            ticks = ticks[zero_index[0]:]  # trim to start from zero
        g.ax_joint.set_xlim(0, max_val)
        g.ax_joint.set_ylim(0, max_val)
    g.ax_joint.set_xticks(ticks)
    g.ax_joint.set_yticks(ticks)  # optional but ensures both are aligned

    # Marginal histograms
    # After g.plot_marginals(...)
    g.plot_marginals(sns.histplot, color="#6a8bef", alpha=0.6, kde=True, linewidth=1.2, bins=35)
    g.ax_marg_x.set_ylim(top=150)  # y-axis for top histogram
    g.ax_marg_y.set_xlim(right=150)  # x-axis for right histogram

    # Labels and title
    if p == "spillage":
        g.set_axis_labels(f"TensorFlow pred {names[i]}", f"PyTorch pred {names[i]}", fontsize=12, fontstyle='italic')
    else:
        g.set_axis_labels(f"PyTorch pred {names[i]}({units[i]})", f"TensorFlow pred {names[i]}({units[i]})",  fontsize=12, fontstyle='italic')
    # g.ax_joint.set_title(f"Parity Plot with Histograms - {p}", fontsize=14)
    g.ax_joint.tick_params(labelsize=14)

    g.ax_joint.text(0.08, 0.92, f"{names[i]}\nMAD (TF vs. PyTorch): {mad_value:.3f}",
                    transform=g.ax_joint.transAxes,
                    ha='left', va='top',
                    fontsize=14, fontstyle='italic')

    # Enable borders (spines) for all axes
    for spine in g.ax_joint.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)

    # for spine in g.ax_marg_x.spines.values():
    #     spine.set_visible(True)
    #     spine.set_linewidth(1.2)

    # for spine in g.ax_marg_y.spines.values():
    #     spine.set_visible(True)
    #     spine.set_linewidth(1.2)
    plt.tight_layout()
    plt.savefig(f"deterministics/pytorch_tf_parity_plot_{p}.png", dpi=500, bbox_inches='tight')
    
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

properties = ["formation_energy_peratom", "ehull", "magmom_outcar", "mbj_bandgap", "spillage", "slme", "Tc_supercon"]
names = ["Formation Energy", "Energy above the Hull", "Magnetic Moment", "Bandgap_mBJ", "Spillage", "SLME", "Tc_Supercon"]
units = ['eV/atom', 'eV/atom', 'μB', 'eV', '', '%', 'K']

for i, p in enumerate(properties):
    # Load predictions

    df_tf = pd.read_csv(f"deterministics/tensorflow/alignn_matbert-base-cased_robo_prop_{p}_pred_otf.csv", index_col=0)
    df_pt = pd.read_csv(f"deterministics/prediction_torch/alignn_matbert-base-cased_robo_prop_{p}_pred_otf.csv")

    # Optional filtering
    if p == "magmom_outcar":
        mask = df_tf["labels"] != 0
        df_tf = df_tf[mask]
        df_pt = df_pt[mask]

    # Add predictions
    df = pd.DataFrame({
        "TF": df_tf["predictions"].astype(float),
        "PT": df_pt["predictions"].astype(float)
    })
    mad_value = np.mean(np.abs(df["TF"] - df["PT"]))

    # --- Bland–Altman Plot ---
    mean_pred = (df["TF"] + df["PT"]) / 2
    diff = df["PT"] - df["TF"]
    mean_diff = np.mean(diff)
    std_diff = np.std(diff)

    if p == "spillage":
        plt.figure(figsize=(5.2, 5))
    else:
        plt.figure(figsize=(5, 5))
    pred_range = np.max(mean_pred) - np.min(mean_pred)
    ylim_max = 0.25 * pred_range

    if p == "spillage":
        plt.figure(figsize=(5.2, 5))
    else:
        plt.figure(figsize=(5, 5))
    plt.scatter(mean_pred, diff, color='#6a8bef', alpha=1, s=30, edgecolors='white', linewidths=0.5, marker='o')
    plt.axhline(mean_diff, color='red', linestyle='--', linewidth=1.5, label=f'Prediction mean difference: {mean_diff:.4f}')
    plt.axhline(mean_diff + 1.96 * std_diff, color="blue", linestyle='--', linewidth=1.5, label='±1.96 SD of mean difference')
    plt.axhline(mean_diff - 1.96 * std_diff, color="blue", linestyle='--', linewidth=1.5)
    plt.ylim(-ylim_max, ylim_max)

    # plt.title(f"Bland–Altman Plot: {p}", fontsize=12)
    if p == "spillage":
         plt.xlabel(f"Mean of TensorFlow and PyTorch predicted {names[i]}", fontsize=12)
    else:
        plt.xlabel(f"Mean of TensorFlow and PyTorch predicted\n {names[i]} ({units[i]})", fontsize=12)
    plt.ylabel("PyTorch Prediction − TensorFlow Prediction ", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(fontsize=11, loc='upper right')
    plt.tight_layout()
    plt.savefig(f"deterministics/tf_pytorch_bland_altman_plot_{p}.png", dpi=500, bbox_inches='tight')
    plt.show()

#### Cross-system Level Comparison

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# properties = ["formation_energy_peratom", "ehull", "magmom_outcar", "mbj_bandgap", "spillage", "slme", "Tc_supercon"]
properties = ["mbj_bandgap"]
# names = ["Formation Energy", "Energy above the Hull", "Magnetic Moment", "Bandgap_mBJ", "Spillage", "SLME", "Tc_Supercon"]
names = ["Bandgap_mBJ"]
# units = ['eV/atom', 'eV/atom', 'μB', 'eV', '', '%', 'K']
units = ['eV']

for i, p in enumerate(properties):
    # Load predictions
    df_cpu = pd.read_csv(f"deterministics/prediction_torch/alignn_matbert-base-cased_robo_prop_mbj_bandgap_pred_otf_20250614_184558_cpu_leia.csv", index_col=0)
    df_gpu = pd.read_csv(f"deterministics/prediction_torch/alignn_matbert-base-cased_robo_prop_mbj_bandgap_pred_otf_20250614_184558_gpu.csv", index_col=0)

    # Optional filtering
    # if p == "magmom_outcar":
    #     mask = df_cpu["labels"] != 0
    #     df_cpu = df_cpu[mask]
    #     df_gpu = df_gpu[mask]

    # Add predictions
    df = pd.DataFrame({
        "CPU": df_cpu["predictions"].astype(float),
        "GPU": df_gpu["predictions"].astype(float)
    })
    mad_value = np.mean(np.abs(df["CPU"] - df["GPU"]))
    tf_mae_value = np.mean(np.abs(df_cpu["predictions"] - df_cpu["labels"]))
    pt_mae_value = np.mean(np.abs(df_gpu["predictions"] - df_gpu["labels"]))  
    print(f"{p} Mean Absolute Deviation (MAD): {mad_value:.4f}")
    print(f"{p} Tensorflow Mean Absolute Error (MAE): {tf_mae_value:.4f}")
    print(f"{p} PyTorch Mean Absolute Error (MAE): {pt_mae_value:.4f}")
    # Create JointGrid
    g = sns.JointGrid(data=df, x="GPU", y="CPU", height=6, ratio=3)

    # Central scatter plot
    g.plot_joint(sns.scatterplot, color="#e36c55", alpha=0.6, s=40, marker='o')

    # Add y = x reference line
    min_val = min(df["CPU"].min(), df["GPU"].min())
    max_val = max(df["CPU"].max(), df["GPU"].max())
    g.ax_joint.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

    g.ax_joint.set_xlim(min_val, max_val)
    g.ax_joint.set_ylim(min_val, max_val)



    # Force identical ticks
    ticks = g.ax_joint.get_xticks()
    if p in ["Tc_supercon", "spillage", "mbj_bandgap", "slme"]:
        # Force first tick to be zero (or start from zero if zero is in the list)
        ticks = np.array(ticks)
        zero_index = np.where(ticks == 0)[0]
        if len(zero_index) > 0:
            ticks = ticks[zero_index[0]:]  # trim to start from zero
        g.ax_joint.set_xlim(0, max_val)
        g.ax_joint.set_ylim(0, max_val)
    g.ax_joint.set_xticks(ticks)
    g.ax_joint.set_yticks(ticks)  # optional but ensures both are aligned

    # Marginal histograms
    # After g.plot_marginals(...)
    g.plot_marginals(sns.histplot, color="#6a8bef", alpha=0.6, kde=True, linewidth=1.2, bins=35)
    g.ax_marg_x.set_ylim(top=150)  # y-axis for top histogram
    g.ax_marg_y.set_xlim(right=150)  # x-axis for right histogram
    # Labels and title
    if p == "spillage":
        g.set_axis_labels(f"GPU predicted {names[i]}", f"CPU predicted {names[i]}", fontsize=12, fontstyle='italic')
    else:
        g.set_axis_labels(f"GPU predicted {names[i]}({units[i]})", f"CPU predicted {names[i]}({units[i]}) with model trained on GPU", fontsize=12, fontstyle='italic')
    # g.ax_joint.set_title(f"Parity Plot with Histograms - {p}", fontsize=14)
    g.ax_joint.tick_params(labelsize=14)

    g.ax_joint.text(0.08, 0.92, f"{names[i]}\nMAD (CPU Inference \nvs. GPU Inference): {mad_value:.8f}",
                    transform=g.ax_joint.transAxes,
                    ha='left', va='top',
                    fontsize=14, fontstyle='italic')

    # Enable borders (spines) for all axes
    for spine in g.ax_joint.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)

    # for spine in g.ax_marg_x.spines.values():
    #     spine.set_visible(True)
    #     spine.set_linewidth(1.2)

    # for spine in g.ax_marg_y.spines.values():
    #     spine.set_visible(True)
    #     spine.set_linewidth(1.2)
    plt.tight_layout()
    plt.savefig(f"deterministics/tf_cpu_gpu_infer_parity_plot_{p}.png", dpi=500, bbox_inches='tight')
    
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# properties = ["formation_energy_peratom", "ehull", "magmom_outcar", "mbj_bandgap", "spillage", "slme", "Tc_supercon"]
properties = ["mbj_bandgap"]
# names = ["Formation Energy", "Energy above the Hull", "Magnetic Moment", "Bandgap_mBJ", "Spillage", "SLME", "Tc_Supercon"]
names = ["Bandgap_mBJ"]
# units = ['eV/atom', 'eV/atom', 'μB', 'eV', '', '%', 'K']
units = ['eV']

for i, p in enumerate(properties):
    # Load predictions
    df_cpu = pd.read_csv(f"deterministics/prediction_torch/alignn_matbert-base-cased_robo_prop_mbj_bandgap_pred_otf_20250614_193231_cpu.csv", index_col=0)
    df_gpu = pd.read_csv(f"deterministics/prediction_torch/alignn_matbert-base-cased_robo_prop_mbj_bandgap_pred_otf_20250614_184558_gpu.csv", index_col=0)

    # Optional filtering
    # if p == "magmom_outcar":
    #     mask = df_cpu["labels"] != 0
    #     df_cpu = df_cpu[mask]
    #     df_gpu = df_gpu[mask]

    # Add predictions
    df = pd.DataFrame({
        "CPU": df_cpu["predictions"].astype(float),
        "GPU": df_gpu["predictions"].astype(float)
    })
    mad_value = np.mean(np.abs(df["CPU"] - df["GPU"]))
    tf_mae_value = np.mean(np.abs(df_cpu["predictions"] - df_cpu["labels"]))
    pt_mae_value = np.mean(np.abs(df_gpu["predictions"] - df_gpu["labels"]))  
    print(f"{p} Mean Absolute Deviation (MAD): {mad_value:.4f}")
    print(f"{p} Tensorflow Mean Absolute Error (MAE): {tf_mae_value:.4f}")
    print(f"{p} PyTorch Mean Absolute Error (MAE): {pt_mae_value:.4f}")
    # Create JointGrid
    g = sns.JointGrid(data=df, x="GPU", y="CPU", height=6, ratio=3)

    # Central scatter plot
    g.plot_joint(sns.scatterplot, color="#e36c55", alpha=0.6, s=40, marker='o')

    # Add y = x reference line
    min_val = min(df["CPU"].min(), df["GPU"].min())
    max_val = max(df["CPU"].max(), df["GPU"].max())
    g.ax_joint.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

    g.ax_joint.set_xlim(min_val, max_val)
    g.ax_joint.set_ylim(min_val, max_val)



    # Force identical ticks
    ticks = g.ax_joint.get_xticks()
    if p in ["Tc_supercon", "spillage", "mbj_bandgap", "slme"]:
        # Force first tick to be zero (or start from zero if zero is in the list)
        ticks = np.array(ticks)
        zero_index = np.where(ticks == 0)[0]
        if len(zero_index) > 0:
            ticks = ticks[zero_index[0]:]  # trim to start from zero
        g.ax_joint.set_xlim(0, max_val)
        g.ax_joint.set_ylim(0, max_val)
    g.ax_joint.set_xticks(ticks)
    g.ax_joint.set_yticks(ticks)  # optional but ensures both are aligned

    # Marginal histograms
    # After g.plot_marginals(...)
    g.plot_marginals(sns.histplot, color="#6a8bef", alpha=0.6, kde=True, linewidth=1.2, bins=35)
    g.ax_marg_x.set_ylim(top=150)  # y-axis for top histogram
    g.ax_marg_y.set_xlim(right=150)  # x-axis for right histogram

    # Labels and title
    if p == "spillage":
        g.set_axis_labels(f"GPU predicted {names[i]}", f"CPU predicted {names[i]}", fontsize=12, fontstyle='italic')
    else:
        g.set_axis_labels(f"GPU predicted {names[i]}({units[i]})", f"CPU predicted {names[i]}({units[i]})", fontsize=12, fontstyle='italic')
    # g.ax_joint.set_title(f"Parity Plot with Histograms - {p}", fontsize=14)
    g.ax_joint.tick_params(labelsize=14)

    g.ax_joint.text(0.08, 0.92, f"{names[i]}\nMAD (CPU vs. GPU): {mad_value:.4f}",
                    transform=g.ax_joint.transAxes,
                    ha='left', va='top',
                    fontsize=14, fontstyle='italic')

    # Enable borders (spines) for all axes
    for spine in g.ax_joint.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)

    # for spine in g.ax_marg_x.spines.values():
    #     spine.set_visible(True)
    #     spine.set_linewidth(1.2)

    # for spine in g.ax_marg_y.spines.values():
    #     spine.set_visible(True)
    #     spine.set_linewidth(1.2)
    plt.tight_layout()
    plt.savefig(f"deterministics/tf_cpu_gpu_parity_plot_{p}.png", dpi=500, bbox_inches='tight')
    
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# properties = ["formation_energy_peratom", "ehull", "magmom_outcar", "mbj_bandgap", "spillage", "slme", "Tc_supercon"]
properties = ["mbj_bandgap"]
# names = ["Formation Energy", "Energy above the Hull", "Magnetic Moment", "Bandgap_mBJ", "Spillage", "SLME", "Tc_Supercon"]
names = ["Bandgap_mBJ"]
# units = ['eV/atom', 'eV/atom', 'μB', 'eV', '', '%', 'K']
units = ['eV']

for i, p in enumerate(properties):
    # Load predictions
    df_leia = pd.read_csv(f"deterministics/prediction_torch/alignn_matbert-base-cased_robo_prop_mbj_bandgap_pred_otf_20250614_184558_gpu.csv", index_col=0)
    df_colab = pd.read_csv(f"deterministics/prediction_torch/alignn_matbert-base-cased_robo_prop_mbj_bandgap_pred_otf_20250616_015946_gpu_colab.csv", index_col=0)

    # Optional filtering
    # if p == "magmom_outcar":
    #     mask = df_leia["labels"] != 0
    #     df_leia = df_leia[mask]
    #     df_colab = df_colab[mask]

    # Add predictions
    df = pd.DataFrame({
        "Machine 1": df_leia["predictions"].astype(float),
        "Machine 3": df_colab["predictions"].astype(float)
    })
    mad_value = np.mean(np.abs(df["Machine 1"] - df["Machine 3"]))
    tf_mae_value = np.mean(np.abs(df_cpu["predictions"] - df_cpu["labels"]))
    pt_mae_value = np.mean(np.abs(df_colab["predictions"] - df_colab["labels"]))  
    print(f"{p} Mean Absolute Deviation (MAD): {mad_value:.4f}")
    print(f"{p} Tensorflow Mean Absolute Error (MAE): {tf_mae_value:.4f}")
    print(f"{p} PyTorch Mean Absolute Error (MAE): {pt_mae_value:.4f}")
    # Create JointGrid
    g = sns.JointGrid(data=df, x="Machine 1", y="Machine 3", height=6, ratio=3)

    # Central scatter plot
    g.plot_joint(sns.scatterplot, color="#e36c55", alpha=0.6, s=40, marker='o')

    # Add y = x reference line
    min_val = min(df["Machine 1"].min(), df["Machine 3"].min())
    max_val = max(df["Machine 1"].max(), df["Machine 3"].max())
    g.ax_joint.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

    g.ax_joint.set_xlim(min_val, max_val)
    g.ax_joint.set_ylim(min_val, max_val)



    # Force identical ticks
    ticks = g.ax_joint.get_xticks()
    if p in ["Tc_supercon", "spillage", "mbj_bandgap", "slme"]:
        # Force first tick to be zero (or start from zero if zero is in the list)
        ticks = np.array(ticks)
        zero_index = np.where(ticks == 0)[0]
        if len(zero_index) > 0:
            ticks = ticks[zero_index[0]:]  # trim to start from zero
        g.ax_joint.set_xlim(0, max_val)
        g.ax_joint.set_ylim(0, max_val)
    g.ax_joint.set_xticks(ticks)
    g.ax_joint.set_yticks(ticks)  # optional but ensures both are aligned

    # Marginal histograms
    # After g.plot_marginals(...)
    g.plot_marginals(sns.histplot, color="#6a8bef", alpha=0.6, kde=True, linewidth=1.2, bins=35)
    # Clip extreme histogram bin heights
    g.ax_marg_x.set_ylim(top=150)  # y-axis for top histogram
    g.ax_marg_y.set_xlim(right=150)  # x-axis for right histogram

    # Labels and title
    if p == "spillage":
        g.set_axis_labels(f"Machine 1 GPU predicted {names[i]}", f"Machine 3 GPU predicted {names[i]}", fontsize=12, fontstyle='italic')
    else:
        g.set_axis_labels(f"Machine 1 GPU predicted {names[i]}({units[i]})", f"Machine 3 GPU predicted {names[i]}({units[i]})", fontsize=12, fontstyle='italic')
    # g.ax_joint.set_title(f"Parity Plot with Histograms - {p}", fontsize=14)
    g.ax_joint.tick_params(labelsize=14)

    g.ax_joint.text(0.08, 0.92, f"{names[i]}\nMAD (Machine 3 \nvs. Machine 1): {mad_value:.4f}",
                    transform=g.ax_joint.transAxes,
                    ha='left', va='top',
                    fontsize=14, fontstyle='italic')

    # Enable borders (spines) for all axes
    for spine in g.ax_joint.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)

    # for spine in g.ax_marg_x.spines.values():
    #     spine.set_visible(True)
    #     spine.set_linewidth(1.2)

    # for spine in g.ax_marg_y.spines.values():
    #     spine.set_visible(True)
    #     spine.set_linewidth(1.2)
    plt.tight_layout()
    plt.savefig(f"deterministics/tf_machine3_machine1_parity_plot_{p}.png", dpi=500, bbox_inches='tight')
    
    plt.show()